In [1]:
%load_ext autoreload
%autoreload 2

import os

os.chdir("../../")

In [2]:
import pandas as pd

dogs_df = pd.read_excel("baskets_dogs_data/dogs.xlsx")
baskets_df = pd.read_excel("baskets_dogs_data/baskets.xlsx")
pairs = dogs_df.columns.tolist()[3:]
pairs

['Pair_20',
 'Pair_21',
 'Pair_22',
 'Pair_23',
 'Pair_24',
 'Pair_34',
 'Pair_35',
 'Pair_37',
 'Pair_43',
 'Pair_45']

In [4]:
baskets_df.columns

Index(['Round', 'Order', 'TargetObjectIndex', 'Pair_20', 'Pair_21', 'Pair_22',
       'Pair_23', 'Pair_24', 'Pair_34', 'Pair_35', 'Pair_37', 'Pair_43',
       'Pair_45'],
      dtype='object')

In [4]:
from src.data.utils import get_conversations

parsed_transcripts_fp = "baskets_dogs_data/llm_parsed_transcripts.csv"


if os.path.exists(parsed_transcripts_fp):
    transcripts_df = pd.read_csv(parsed_transcripts_fp)
    print("Loaded existing parsed transcripts.")

else:

    transcripts = []

    for object_category in ["baskets", "dogs"]:
        if object_category == "baskets":
            df = baskets_df
        else:
            df = dogs_df
        
        for round_ix in range(1, 5):
            
            for pair in pairs:
                transcript, target_object_indices = get_conversations(
                    df, round_ix, pair, 
                    return_entire_transcript=True
                )
                transcripts.append([object_category, round_ix, pair, transcript])

    transcripts_df = pd.DataFrame(transcripts, 
                                columns=["object_category", "round_ix", "pair_id", "transcript"])
    transcripts_df.to_csv(parsed_transcripts_fp, index=False)
    print("Saved parsed transcripts.")

transcripts_df.shape

Loaded existing parsed transcripts.


(80, 5)

In [10]:
transcripts_df.head()

,object_category,round_ix,pair_id,transcript
0,baskets,1,Pair_20,"D: Alright, the first one is a, uh--is a very ..."
1,baskets,1,Pair_21,"D: Ok, uh…\n You ready?\n \n M: Yup\n \n D: (l..."
2,baskets,1,Pair_22,"D: Alright, hello?\n Um, the first one is--is...."
3,baskets,1,Pair_23,"D: Ok, Parastoo you see the...sort of...oblong..."
4,baskets,1,Pair_24,"D: K, uh, the first basket is longer than the ..."


In [5]:
from string import Template


prompt_tmp = ''' 
You are given a verbatim conversation transcript between two participants \
who are playing a game in which they identify target objects from a set of images.

Your task is to segment the transcript into exactly 10 parts, \
where each part corresponds to a distinct target object discussed in the conversation.

### Requirements

- Each part must be copied verbatim from the transcript.
- Do not paraphrase, summarize, edit, or add any text.
- Each transcript segment must be contiguous (no skipping or reordering lines).
- Every part should focus primarily on one specific target object.
- Do not include explanations, labels, or commentary outside of the transcript text.

### Output Format

Return the result as a valid JSON array of objects. Each object must follow this exact structure:

{
  "part_index": <integer starting from 1>,
  "transcript_segment": "<verbatim transcript segment>"
}

- part_index must range from 1 to 10, in order.
- The final output must be strictly valid JSON and parsable without errors.
- Do not include any text before or after the JSON array.

### Transcript

$transcript

### Your Response

Return only the JSON array described above.
'''

prompt_tmp= Template(prompt_tmp)
print(prompt_tmp.template)

 
You are given a verbatim conversation transcript between two participants who are playing a game in which they identify target objects from a set of images.

Your task is to segment the transcript into exactly 10 parts, where each part corresponds to a distinct target object discussed in the conversation.

### Requirements

- Each part must be copied verbatim from the transcript.
- Do not paraphrase, summarize, edit, or add any text.
- Each transcript segment must be contiguous (no skipping or reordering lines).
- Every part should focus primarily on one specific target object.
- Do not include explanations, labels, or commentary outside of the transcript text.

### Output Format

Return the result as a valid JSON array of objects. Each object must follow this exact structure:

{
  "part_index": <integer starting from 1>,
  "transcript_segment": "<verbatim transcript segment>"
}

- part_index must range from 1 to 10, in order.
- The final output must be strictly valid JSON and parsab

### Test

In [19]:
transcript = transcripts_df.iloc[0].transcript
prompt = prompt_tmp.substitute(transcript=transcript)
print(prompt)

 
You are given a verbatim conversation transcript between two participants who are playing a game in which they identify target objects from a set of images.

Your task is to segment the transcript into exactly 10 parts, where each part corresponds to a distinct target object discussed in the conversation.

### Requirements

- Each part must be copied verbatim from the transcript.
- Do not paraphrase, summarize, edit, or add any text.
- Each transcript segment must be contiguous (no skipping or reordering lines).
- Every part should focus primarily on one specific target object.
- Do not include explanations, labels, or commentary outside of the transcript text.

### Output Format

Return the result as a valid JSON array of objects. Each object must follow this exact structure:

{
  "part_index": <integer starting from 1>,
  "transcript_segment": "<verbatim transcript segment>"
}

- part_index must range from 1 to 10, in order.
- The final output must be strictly valid JSON and parsab

In [20]:
from src.llms.openai import get_response

_, response = get_response(
    prompt,
    model_name="gpt-5-nano"
)
print(response)

[
  {
    "part_index": 1,
    "transcript_segment": "D: Alright, the first one is a, uh--is a very long basket\n \n M: Mm-hmm\n \n D: Uh, it's rectangular in shape\n It's a light tan\n Weaved\n Uh, there's--that's about it, in that, uh--the description of that\n It's uh...I'm facing it with the handle coming over the top\n \n M: It's right in the middle, the handle?\n \n D: Yes\n \n M: Ok\n \n D: Uh…I don't know, you probably have that one\n \n M: Yeah *I got it*"
  },
  {
    "part_index": 2,
    "transcript_segment": "D: *We* go with number two here\n Number two's got a huge handle, and it's a very small basket\n It's, uh--got like red, and uh, brown and green weaving in it?\n Huge handle\n Right in the middle, facing towards, uh--I mean I can see right through the handle\n \n M: Mm-hmm\n It's like a really small, like...\n \n D: Very tiny basket compared to the handle\n \n M: Ok\n Alright I think I got that one"
  },
  {
    "part_index": 3,
    "transcript_segment": "D: Ok, number

### Deployment

In [ ]:
from tqdm import tqdm

parsed_transcripts_fp = "baskets_dogs_data/llm_parsed_transcripts.csv"

for ix, row in tqdm(transcripts_df.iterrows(), total=len(transcripts_df)):
    transcript = row.transcript
    prompt = prompt_tmp.substitute(transcript=transcript)

    if not pd.isna(row.get("model_response")):
        print(f"Skipping index {ix} as it already has a response.")
        continue

    _, response = get_response(
        prompt,
        model_name="gpt-5-nano"
    )
    transcripts_df.at[ix, "model_response"] = response
    transcripts_df.to_csv(parsed_transcripts_fp, index=False)

  0%|          | 0/80 [00:00<?, ?it/s]

100%|██████████| 80/80 [2:22:28<00:00, 106.85s/it]  


In [35]:
import re
import json


def parse_llm_response(response):
    response = response.replace("json", "").strip()
    try:
        parsed = json.loads(response)
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e}")
        parsed = []
    return parsed

# def parse_llm_response(response):
#     responses = re.findall(r"\{[^\}]+\}", response)
#     outputs = []
#     if not responses:
#         return outputs
    
#     for response in responses:
    
#         try:
#             parsed = json.loads(response)
            
#         except json.JSONDecodeError:
#             parsed = {}
        
#         outputs.extend(parsed)

#     return outputs

In [36]:
transcripts_df["parsed_response"] = transcripts_df["model_response"].apply(parse_llm_response)
transcripts_df.head()

JSONDecodeError: Invalid control character at: line 4 column 42 (char 68)


,object_category,round_ix,pair_id,transcript,model_response,parsed_response
0,baskets,1,Pair_20,"D: Alright, the first one is a, uh--is a very ...","[\n {\n ""part_index"": 1,\n ""transcript_...","[{'part_index': 1, 'transcript_segment': 'D: A..."
1,baskets,1,Pair_21,"D: Ok, uh…\n You ready?\n \n M: Yup\n \n D: (l...","[\n {\n ""part_index"": 1,\n ""transcript_...","[{'part_index': 1, 'transcript_segment': 'D: O..."
2,baskets,1,Pair_22,"D: Alright, hello?\n Um, the first one is--is....","[\n {\n ""part_index"": 1,\n ""transcript_...","[{'part_index': 1, 'transcript_segment': 'D: A..."
3,baskets,1,Pair_23,"D: Ok, Parastoo you see the...sort of...oblong...","[\n {\n ""part_index"": 1,\n ""transcript_...","[{'part_index': 1, 'transcript_segment': 'D: O..."
4,baskets,1,Pair_24,"D: K, uh, the first basket is longer than the ...","[\n {\n ""part_index"": 1,\n ""transcript_...","[{'part_index': 1, 'transcript_segment': 'D: K..."


In [33]:
sub = transcripts_df[transcripts_df.parsed_response.apply(len) != 10]
sub

,object_category,round_ix,pair_id,transcript,model_response,parsed_response
43,dogs,1,Pair_23,"D: Ok (laughs)\nOk, this dog is standing upri...","[\n {\n ""part_index"": 1,\n ""transcript_...",[]


In [34]:
print(sub.model_response.iloc[0])

[
  {
    "part_index": 1,
    "transcript_segment": "D: Ok (laughs)
Ok, this dog is standing upright
(laughs)

M: Ok

D: Um, he's got a patch of black around his stomach and back area, and his legs are more like a camel color

M: Um, like a orangish color?

D: Yeah

M: Um, like towards his tail, is it like a y--uh--yellowish…color?

D: Yeah, but he has like a black tip on his tail

M: Yup got it, ok"
  },
  {
    "part_index": 2,
    "transcript_segment": "D: Ok, number two--he's--
(cell phone rings)
He's furry
M: He's furry
Ok

D: Um, (cell phone rings) and he's--his legs--he's--he's sort of like turning towards his butt

M: (laughs) 
Ok

D: And, um, he's got white legs, and his--he's got a patch of black on his--his--his butt, too

M: He's got white legs, you said?

D: Yeah, like he looks like he's got maybe like, a little bit of fur
Like maybe two inches of fur, that’s it

M: White legs
Does he look like a wolf?
Kinda?

D: N--no
He's looking--um, the way he's facing, *he--*

M: *He

In [29]:
# print(transcripts_df.iloc[1].model_response)

In [30]:
print(json.dumps(transcripts_df.iloc[1].parsed_response, indent=4))

[
    {
        "part_index": 1,
        "transcript_segment": "D: Ok, uh\u2026\n You ready?\n \n M: Yup\n \n D: (laughs)\n Ok, the first one is a long basket\n It looks like a rectangle\n It's probably the longest one in the picture\n It's square in shape\n Rectangular in shape\n \n M: Ok\n \n D: You got it?\n \n M: Yup"
    },
    {
        "part_index": 2,
        "transcript_segment": "D: Ok\n The second one has a very big handle on it, and a very small basket in a pyramidal shape\n And a really big, circular looking handle\n \n M: Big circular looking handle\n Alright\n \n D: Big circular handle\n It's probably got the longest base from the basket to the hang--uh, handle\n \n M: Ok"
    },
    {
        "part_index": 3,
        "transcript_segment": "D: Uh, third one has, uh, criss-crossing--oh, they all have criss-crossing weaves, but this one has probably the one with the most open space in between each\n \n M: Alright\n Got it\n \n D: And it's got a circular handle\n \n M: Alri